In [1]:
import os
# os.environ["HF_HOME"] = "/home/seungwoochoi/data/huggingface/cache"
from tqdm import tqdm
import torch
import torch.nn as nn
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from FlagEmbedding import BGEM3FlagModel
import numpy as np
device = "mps"
np.set_printoptions(threshold=np.inf)
import matplotlib.pyplot as plt

/opt/miniconda3/envs/axis_rag/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
embedding_model = BGEM3FlagModel('BAAI/bge-m3', devices=device)

Fetching 30 files: 100%|██████████| 30/30 [00:00<00:00, 293993.27it/s]


In [155]:
import pandas as pd 
ambiguous_words_df = pd.read_csv("data/ambiguous_words_paired.csv")

In [238]:
print(list(ambiguous_words_df['Meaning 1']))
# embedding_model.encode(list(ambiguous_words_df['Meaning 1']))
meaning1_embedding = embedding_model.encode(list(ambiguous_words_df['Meaning 1']))['dense_vecs']
meaning2_embedding = embedding_model.encode(list(ambiguous_words_df['Meaning 2']))['dense_vecs']
meaning1_embedding_train = meaning1_embedding[:15]
meaning1_embedding_test = meaning1_embedding[15:]
meaning2_embedding_train = meaning2_embedding[:15]
meaning2_embedding_test = meaning2_embedding[15:]

def1_embedding = embedding_model.encode(list(ambiguous_words_df['Def1']))['dense_vecs']
def1_embedding_test = def1_embedding[15:]


['Bank, a financial institution', 'Bat, a flying mammal', 'Bark, the sound a dog makes', 'Spring, a season', 'Jam, a fruit preserve', 'Right, correct', 'Light, illumination', 'Match, a competition', 'Watch, to observe', 'Rock, a stone', 'Ring, a piece of jewelry', 'Seal, a marine animal', 'Current, the flow of water', 'File, a folder for documents', 'Nail, a fastener', 'Can, a container', 'Well, a water source', 'Point, a sharp tip', 'Trip, a journey', 'Row, a line of items']


In [254]:
amb_hadamard = meaning1_embedding_train * meaning2_embedding_train #여기 값들은 낮아야 함. 값이 큰 dimension들은 제거대상

syntax_dims = set()
for i in range(amb_hadamard.shape[0]):
    [syntax_dims.add(x.item()) for x in list(np.where(amb_hadamard[i] > 0.003)[0])]
len(syntax_dims)

268

In [ ]:
synonyms_df = pd.read_csv("data/synonyms_100.csv")

synonym1_embedding = embedding_model.encode(list(synonyms_df['Word']))['dense_vecs']
synonym2_embedding = embedding_model.encode(list(synonyms_df['Synonym']))['dense_vecs']

synonyms_df = pd.read_csv("data/same_mean_sents2.csv")
synonym1_embedding = embedding_model.encode(list(synonyms_df['Word']))['dense_vecs']
synonym2_embedding = embedding_model.encode(list(synonyms_df['Synonym']))['dense_vecs']

sym_hadamard = synonym1_embedding * synonym2_embedding #여기 값들은 낮아야 함. 값이 큰 dimension들이 포함 대상
semantic_dims = set()

for i in range(sym_hadamard.shape[0]):
    [semantic_dims.add(x.item()) for x in list(np.where(sym_hadamard[i] > 0.007)[0])]

len(semantic_dims)

67

In [258]:
semantic_dims = semantic_dims - syntax_dims

In [259]:
print(len(semantic_dims))
print(list(semantic_dims))

11
[355, 388, 809, 906, 236, 558, 463, 85, 278, 598, 667]


In [260]:

meaning1_embedding_filtered = meaning1_embedding_test[:, list(semantic_dims)]
meaning2_embedding_filtered = meaning2_embedding_test[:, list(semantic_dims)]
def1_embedding_filtered = def1_embedding_test[:, list(semantic_dims)]

print(def1_embedding_filtered.shape)

print(np.mean((meaning1_embedding_filtered @ meaning2_embedding_filtered.T).diagonal()))
print(np.mean((meaning1_embedding_filtered @ def1_embedding_filtered.T).diagonal()))

(5, 11)
0.00848
0.00917


# Synonyms

In [154]:
from collections import Counter
syntax_dims = []


for j in range(hadamard.shape[0]):
    [syntax_dims.append(x.item()) for x in list(np.where(hadamard[j] > 0.005)[0])]


syntax_dims_list = list(syntax_dims)
print(len(syntax_dims_list))
dim_counter = Counter(syntax_dims_list)

syntax_dims_list = [x for x in dim_counter.keys() if dim_counter[x] > 15]
print(len(syntax_dims_list))
# meaning1_embedding_filtered = np.delete(meaning1_embedding, syntax_dims_list, axis=1)
# meaning2_embedding_filtered = np.delete(meaning2_embedding, syntax_dims_list, axis=1)
# def1_embedding_filtered = np.delete(def1_embedding, syntax_dims_list, axis=1)


meaning1_embedding_filtered = meaning1_embedding[:, syntax_dims_list]
meaning2_embedding_filtered = meaning1_embedding[:, syntax_dims_list]
def1_embedding_filtered = meaning1_embedding[:, syntax_dims_list]

if(np.mean((meaning1_embedding_filtered @ meaning2_embedding_filtered.T).diagonal()) < np.mean((meaning1_embedding_filtered @ def1_embedding_filtered.T).diagonal())): 
    print(np.mean((meaning1_embedding_filtered @ meaning2_embedding_filtered.T).diagonal()))
    print(np.mean((meaning1_embedding_filtered @ def1_embedding_filtered.T).diagonal()))
    

585
9
